# APS360 Progress Report: human-like chess move prediction by rating band (v2)
Run all cells top to bottom on a **GPU Colab runtime** (Runtime > Change runtime type > T4 GPU).
Expect roughly 45-60 min on a T4 with the default sizes (most of it is streaming games and training
3 seeds x 2 variants). To go faster, lower `GAMES_PER_BAND`, `N_SEEDS` or `EPOCHS` in the config cell.
At the end it zips figures, `results.json`, `numbers.tex` and model weights into `progress_outputs.zip`.

**If Colab disconnects or restarts:** allow the Google Drive prompt in the config cell. The datasets and
each trained model are saved to `MyDrive/aps360_ckpt` as soon as they finish, so after a restart just
reconnect (GPU) and Run all again: finished steps load in seconds and it carries on from there.
Keep the browser tab open; free Colab drops sessions that look idle.

Changes from v1:
* **Primary model**: residual CNN with a fully convolutional from-to policy head (about 2.0M parameters).
  The v1 head put 97% of its 8.6M parameters in one fully connected layer and overfit after 3 epochs.
  The v1 network is still trained on the same data as a comparison.
* **More data, spread over more time**: 8,000 games per band (was 6,000). Each eligible game is kept with
  probability `KEEP_PROB`, so the sample covers about 4x more hours of the month.
* **Is the rating band used?** 3 seeds for band vs. no-band, a band-X-tested-on-band-Y matrix, the
  average (not just the largest) band effect, and a game-level bootstrap CI on the band gain.
* **Player leakage check**: hashed player IDs (no usernames kept) to score test games whose players never
  appear in training.
* Report-sized figures (5.5 in wide, 7-8 pt fonts), legends outside the bars, chosen epoch marked.

Pipeline: stream one Lichess monthly dump -> filter -> encode boards -> split by game ->
baselines (capture heuristic, linear model) -> CNNs -> analysis -> figures -> test on a *different month*.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "python-chess", "zstandard"], check=False)

In [ ]:
import os, io, re, json, time, random, shutil, hashlib
import numpy as np
import chess, chess.pgn
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

SYNTHETIC = os.environ.get("SYNTHETIC", "0") == "1"   # only used for local testing of the code
TRAIN_MONTH, TEST_MONTH = "2025-01", "2025-02"        # test month is never used for training or tuning
GAMES_PER_BAND = 8000 if not SYNTHETIC else 40
TEST_GAMES_PER_BAND = 1000 if not SYNTHETIC else 15
KEEP_PROB = 0.25 if not SYNTHETIC else 1.0           # keep each eligible game with this prob -> wider time span
POS_PER_GAME = 12
BAND_EDGES = [1100, 1300, 1500, 1700, 1900, 2100]
NB = 5                 # number of rating bands
MAXL = 128             # max legal moves stored per position
MIN_PLY, MIN_CLOCK = 10, 30
EPOCHS = 10 if not SYNTHETIC else 2
N_SEEDS = 3 if not SYNTHETIC else 2                  # seeds for the primary model (band and no-band)
BATCH = 512
LR, WD = 1e-3, 1e-4
CH, NBLOCKS = 128, 6   # primary model width / number of residual blocks
SEED = 0
OUT = "outputs"
os.makedirs(OUT, exist_ok=True)

# Checkpoints: the datasets and every trained model are saved as soon as they are done. If Colab
# disconnects or restarts, just "Run all" again and finished steps are loaded instead of redone.
# On Colab they go to Google Drive (survives restarts); elsewhere to a local folder.
CKPT_ROOT = "ckpt_synthetic" if SYNTHETIC else "ckpt"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    CKPT_ROOT = "/content/drive/MyDrive/aps360_ckpt"
except Exception as e:
    print("Google Drive not mounted, checkpoints stay local:", type(e).__name__)
# Changing a data/training setting starts a fresh checkpoint folder, so stale results are never reused.
CKPT = os.path.join(CKPT_ROOT, f"g{GAMES_PER_BAND}_t{TEST_GAMES_PER_BAND}_k{KEEP_PROB}_e{EPOCHS}_c{CH}x{NBLOCKS}")
os.makedirs(CKPT, exist_ok=True)
print("checkpoints:", CKPT)
T_START = time.time()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = device.type == "cuda"
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("device:", device, "| synthetic:", SYNTHETIC)
BAND_NAMES = [f"{BAND_EDGES[i]}-{BAND_EDGES[i+1]}" for i in range(NB)]

plt.rcParams.update({"font.size": 8, "axes.titlesize": 8, "axes.labelsize": 8, "xtick.labelsize": 7,
                     "ytick.labelsize": 7, "legend.fontsize": 7, "figure.dpi": 100, "savefig.dpi": 300,
                     "axes.spines.top": False, "axes.spines.right": False})
FIG_W = 5.5            # ICLR text width in inches: include figures with width=\textwidth
C_HEUR, C_LIN, C_V1, C_NC, C_CNN = "#9e9e9e", "#4c78a8", "#b279a2", "#f58518", "#2ca02c"

## 1. Data: stream, filter, encode

In [ ]:
def lichess_lines(month):
    """Stream a Lichess monthly dump line by line. Closing the generator stops the download."""
    import requests, zstandard
    url = f"https://database.lichess.org/standard/lichess_db_standard_rated_{month}.pgn.zst"
    r = requests.get(url, stream=True, timeout=60)
    r.raise_for_status()
    try:
        reader = zstandard.ZstdDecompressor().stream_reader(r.raw)
        for line in io.TextIOWrapper(reader, encoding="utf-8"):
            yield line
    finally:
        r.close()

def synthetic_lines(seed, n_games=4000):
    """Random legal games with fake ratings/clocks. ONLY for testing that the code runs."""
    rng = random.Random(seed)
    for k in range(n_games):
        g = chess.pgn.Game(); b = g.board(); node = g
        w = rng.randint(1000, 2200); bl = w + rng.randint(-250, 250)
        g.headers.update({"Event": "Rated Rapid game", "WhiteElo": str(w), "BlackElo": str(bl),
                          "White": f"p{rng.randint(0, 300)}", "Black": f"p{rng.randint(0, 300)}",
                          "UTCDate": "2025.01.01", "UTCTime": f"{k // 3600 % 24:02d}:{k // 60 % 60:02d}:{k % 60:02d}",
                          "Termination": "Normal", "Result": "1-0"})
        clock = [600.0, 600.0]
        for ply in range(rng.randint(30, 90)):
            moves = list(b.legal_moves)
            if not moves: break
            mv = rng.choice(moves); b.push(mv); node = node.add_variation(mv)
            c = ply % 2; clock[c] -= rng.uniform(1, 8); node.set_clock(max(clock[c], 0))
            if b.is_game_over(): break
        exp = chess.pgn.StringExporter(headers=True, variations=False, comments=True)
        for line in (g.accept(exp) + "\n\n").splitlines(keepends=True):
            yield line

def iter_games(lines):
    buf = []
    for line in lines:
        if line.startswith("[Event ") and buf:
            yield "".join(buf); buf = []
        buf.append(line)
    if buf: yield "".join(buf)

TAG = re.compile(r'\[(\w+) "(.*?)"\]')
def headers(txt):
    return dict(TAG.findall(txt.split("\n\n", 1)[0]))

def band_of(h):
    """Return rating band index 0..4 or None if the game is filtered out."""
    ev = h.get("Event", "")
    if "Rated" not in ev or "Rapid" not in ev: return None          # rated rapid only
    if h.get("Termination") != "Normal": return None                # no time forfeits / abandons
    if "BOT" in (h.get("WhiteTitle"), h.get("BlackTitle")): return None   # humans only
    try:
        w, b = int(h["WhiteElo"]), int(h["BlackElo"])
    except (KeyError, ValueError):
        return None
    if abs(w - b) > 200: return None                                # similar-strength players
    m = (w + b) / 2
    if m < BAND_EDGES[0] or m >= BAND_EDGES[-1]: return None
    return int((m - BAND_EDGES[0]) // 200)

def player_id(name):
    """Stable 63-bit hash of a username. Only the hash is stored (used for the leakage check)."""
    return int.from_bytes(hashlib.sha1(name.encode()).digest()[:8], "little") & ((1 << 63) - 1)

PIECES = [chess.PAWN, chess.KNIGHT, chess.BISHOP, chess.ROOK, chess.QUEEN, chess.KING]
def encode(board):
    """17 x 8 x 8 uint8. Board must be from the mover's view (mover = WHITE, moving up the rows)."""
    x = np.zeros((17, 8, 8), np.uint8)
    for sq, p in board.piece_map().items():
        x[PIECES.index(p.piece_type) + (0 if p.color == chess.WHITE else 6), sq // 8, sq % 8] = 1
    x[12] = board.has_kingside_castling_rights(chess.WHITE)
    x[13] = board.has_queenside_castling_rights(chess.WHITE)
    x[14] = board.has_kingside_castling_rights(chess.BLACK)
    x[15] = board.has_queenside_castling_rights(chess.BLACK)
    if board.ep_square is not None:
        x[16, board.ep_square // 8, board.ep_square % 8] = 1
    return x

def process_game(txt, rng):
    """Return list of (x, label, legal_idx_array, ply) sampled from one game."""
    game = chess.pgn.read_game(io.StringIO(txt))
    if game is None: return []
    moves, ok = [], []
    node = game; ply = 0
    while node.variations:
        node = node.variation(0)
        moves.append(node.move)
        clk = node.clock()   # mover's clock AFTER the move
        ok.append(ply >= MIN_PLY and clk is not None and clk >= MIN_CLOCK)
        ply += 1
    elig = [i for i, o in enumerate(ok) if o]
    if not elig: return []
    pick = set(rng.sample(elig, min(POS_PER_GAME, len(elig))))
    out = []; board = game.board()
    for i, mv in enumerate(moves):
        if i in pick:
            bd = board if board.turn == chess.WHITE else board.mirror()
            m = mv if board.turn == chess.WHITE else chess.Move(chess.square_mirror(mv.from_square),
                                                                chess.square_mirror(mv.to_square))
            legal = sorted({l.from_square * 64 + l.to_square for l in bd.legal_moves})
            if 2 <= len(legal) <= MAXL:      # drop forced moves
                out.append((encode(bd), m.from_square * 64 + m.to_square, np.array(legal, np.int16), i))
        board.push(mv)
    return out

def build_dataset(month, games_per_band, seed, max_scan=10_000_000):
    rng = random.Random(seed)
    src = synthetic_lines(seed) if SYNTHETIC else lichess_lines(month)
    quota = [0] * NB; X, Y, L, B, G, P, WP, BP = [], [], [], [], [], [], [], []
    scanned = 0; kept_games = 0; t0 = time.time(); first_ts = last_ts = None
    for txt in iter_games(src):
        scanned += 1
        h = headers(txt)
        b = band_of(h)
        if b is not None and quota[b] < games_per_band and rng.random() < KEEP_PROB:
            rows = process_game(txt, rng)
            if rows:
                quota[b] += 1; kept_games += 1
                ts = f"{h.get('UTCDate', '?')} {h.get('UTCTime', '?')}"
                first_ts = first_ts or ts; last_ts = ts
                wp, bp = player_id(h.get("White", "?")), player_id(h.get("Black", "?"))
                for (x, y, l, ply) in rows:
                    X.append(x); Y.append(y); B.append(b); G.append(kept_games); P.append(ply)
                    WP.append(wp); BP.append(bp)
                    pad = np.full(MAXL, -1, np.int16); pad[:len(l)] = l; L.append(pad)
        if scanned % 100000 == 0:
            print(f"  scanned {scanned:,} | kept games per band {quota} | {time.time()-t0:.0f}s")
        if min(quota) >= games_per_band or scanned >= max_scan: break
    src.close()
    stats = dict(scanned=scanned, kept=kept_games, games_per_band=quota, first_game_utc=first_ts, last_game_utc=last_ts)
    d = dict(X=np.stack(X), y=np.array(Y, np.int16), legal=np.stack(L), band=np.array(B, np.int8),
             gid=np.array(G, np.int32), ply=np.array(P, np.int16), wp=np.array(WP, np.int64), bp=np.array(BP, np.int64))
    print(f"{month}: scanned {scanned:,} games, kept {kept_games:,}, {len(d['y']):,} positions, "
          f"games from {first_ts} to {last_ts} UTC, {time.time()-t0:.0f}s")
    return d, stats

def cached_dataset(name, month, games_per_band, seed):
    """Build once, then load from the checkpoint folder on every later run."""
    path = os.path.join(CKPT, f"{name}.npz")
    if os.path.exists(path):
        z = np.load(path, allow_pickle=False)
        d = {k: z[k] for k in z.files if k != "stats"}
        stats = json.loads(str(z["stats"]))
        print(f"loaded {name} from checkpoint: {len(d['y']):,} positions")
        return d, stats
    d, stats = build_dataset(month, games_per_band, seed)
    np.savez(path + ".tmp.npz", stats=json.dumps(stats), **d)    # write then rename: no half-written files
    os.replace(path + ".tmp.npz", path)
    return d, stats

train_raw, train_stats = cached_dataset("dataset_train", TRAIN_MONTH, GAMES_PER_BAND, SEED)

## 2. Split by game, leakage check, and a look at the cleaned data

In [ ]:
def split_by_game(d, seed=SEED):
    rng = np.random.RandomState(seed)
    gids = np.unique(d["gid"]); rng.shuffle(gids)
    n = len(gids); tr, va = gids[:int(.8 * n)], gids[int(.8 * n):int(.9 * n)]
    g = d["gid"]
    return (np.isin(g, tr), np.isin(g, va), ~np.isin(g, np.concatenate([tr, va])))

mtr, mva, mte = split_by_game(train_raw)
def take(d, m): return {k: v[m] for k, v in d.items()}
tr, va, te = take(train_raw, mtr), take(train_raw, mva), take(train_raw, mte)
print("positions train/val/test:", len(tr["y"]), len(va["y"]), len(te["y"]))
assert not (set(tr["gid"]) & set(te["gid"])) and not (set(tr["gid"]) & set(va["gid"]))   # no game leakage

# Player-level leakage: the split is by game, so a player can appear in both train and test.
train_players = np.unique(np.concatenate([tr["wp"], tr["bp"]]))
def seen_mask(d):
    return np.isin(d["wp"], train_players) | np.isin(d["bp"], train_players)
te_seen = seen_mask(te)
print(f"test positions whose game has a player who also appears in training: {te_seen.mean():.1%}")

pos_per_band = [int((train_raw["band"] == b).sum()) for b in range(NB)]
games_per_band = train_stats["games_per_band"]
avg_legal = float((train_raw["legal"] >= 0).sum(1).mean())
print("games per band:", games_per_band, "| positions per band:", pos_per_band, "| avg legal moves:", round(avg_legal, 1))

In [ ]:
GLYPH_OWN, GLYPH_OPP = "♙♘♗♖♕♔", "♟♞♝♜♛♚"
def draw_board(ax, x, moves=(), fs=11):
    """x: 17x8x8 from mover's view (mover at bottom). moves: list of (idx, color, style)."""
    for r in range(8):
        for c in range(8):
            ax.add_patch(plt.Rectangle((c, r), 1, 1, color="#f0d9b5" if (r + c) % 2 == 1 else "#b58863"))
    for k in range(12):
        for r, c in zip(*np.nonzero(x[k])):
            g = GLYPH_OWN[k] if k < 6 else GLYPH_OPP[k - 6]
            ax.text(c + .5, r + .45, g, ha="center", va="center", fontsize=fs, color="black")
    for idx, col, ls in moves:
        f, t = idx // 64, idx % 64
        ax.annotate("", xy=(t % 8 + .5, t // 8 + .5), xytext=(f % 8 + .5, f // 8 + .5),
                    arrowprops=dict(arrowstyle="-|>", color=col, lw=1.8, ls=ls, alpha=.9))
    for i in range(8):   # file / rank labels from the mover's view
        ax.text(i + .5, -.3, "abcdefgh"[i], ha="center", va="center", fontsize=5.5)
        ax.text(-.3, i + .5, str(i + 1), ha="center", va="center", fontsize=5.5)
    ax.set_xlim(-.5, 8); ax.set_ylim(-.5, 8); ax.set_aspect("equal"); ax.axis("off")

def sq(i): return chess.square_name(i)
def mv_name(idx): return sq(idx // 64) + sq(idx % 64)
def band_ticks(ax, axis="x", fs=6.5):
    """Rotated one-line band labels so they never collide."""
    if axis == "x":
        ax.set_xticks(range(NB)); ax.set_xticklabels(BAND_NAMES, rotation=35, ha="right", rotation_mode="anchor", fontsize=fs)
    else:
        ax.set_yticks(range(NB)); ax.set_yticklabels(BAND_NAMES, fontsize=fs)

fig, ax = plt.subplots(1, 3, figsize=(FIG_W, 1.9), gridspec_kw=dict(width_ratios=[1.1, 1.1, 0.9]))
ax[0].bar(range(NB), [p / 1000 for p in pos_per_band], color=C_LIN)
band_ticks(ax[0])
ax[0].set_xlabel("rating band"); ax[0].set_ylabel("positions (thousands)")
ax[0].set_title("(a) Positions per band")
ax[1].hist(train_raw["ply"], bins=np.arange(MIN_PLY, 161, 5), color=C_NC)
ax[1].set_xlabel("ply (half-move) of sampled position"); ax[1].set_ylabel("positions")
ax[1].set_title("(b) Where positions come from")
i0 = 0
draw_board(ax[2], tr["X"][i0], [(int(tr["y"][i0]), C_CNN, "-")], fs=9)
ax[2].set_title(f"(c) Sample ({BAND_NAMES[int(tr['band'][i0])]})\nplayed {mv_name(int(tr['y'][i0]))}", fontsize=7)
plt.tight_layout(pad=0.3); plt.savefig(f"{OUT}/fig_data.png", bbox_inches="tight"); plt.savefig(f"{OUT}/fig_data.pdf", bbox_inches="tight"); plt.close()

## 3. Models, training loop, evaluation

In [ ]:
def to_dev(d):
    # legal moves stay int16 (4x smaller than int64); legal_mask converts one batch at a time
    return dict(X=torch.from_numpy(d["X"]).to(device), y=torch.from_numpy(d["y"].astype(np.int64)).to(device),
                legal=torch.from_numpy(d["legal"]).to(device),
                band=torch.from_numpy(d["band"].astype(np.int64)).to(device))
Dtr, Dva, Dte = to_dev(tr), to_dev(va), to_dev(te)
# Free host-memory copies that are no longer needed (the test set keeps its arrays for the figures).
for d in (train_raw, tr, va):
    d.pop("X", None); d.pop("legal", None)
import gc; gc.collect()

def make_input(Xb, bandb, cond=True):
    x = Xb.float()
    oh = F.one_hot(bandb, NB).float()[:, :, None, None].expand(-1, -1, 8, 8) if cond \
        else torch.zeros(len(x), NB, 8, 8, device=x.device)
    return torch.cat([x, oh], 1)

def legal_mask(legal):
    idx = legal.to(torch.long, copy=True); idx[idx < 0] = 4096
    m = torch.full((len(idx), 4097), float("-inf"), device=idx.device)
    m.scatter_(1, idx, 0.0)
    return m[:, :4096]

def logits_for(model, X, band, legal, cond=True):
    with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
        out = model(make_input(X, band, cond))
    return out.float() + legal_mask(legal)      # masking and softmax always in fp32

class ResBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.c1 = nn.Conv2d(ch, ch, 3, padding=1, bias=False); self.b1 = nn.BatchNorm2d(ch)
        self.c2 = nn.Conv2d(ch, ch, 3, padding=1, bias=False); self.b2 = nn.BatchNorm2d(ch)
    def forward(self, x):
        y = F.relu(self.b1(self.c1(x)))
        return F.relu(x + self.b2(self.c2(y)))

class ChessResNet(nn.Module):
    """Primary model. Residual conv tower + fully convolutional policy head.
    The head outputs 64 channels per square: channel t at square f is the logit of the move f -> t.
    After the tower every square's features see the whole board (receptive field > 8x8)."""
    def __init__(self, cin=17 + NB, ch=CH, nblocks=NBLOCKS):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(cin, ch, 3, padding=1, bias=False), nn.BatchNorm2d(ch), nn.ReLU())
        self.tower = nn.Sequential(*[ResBlock(ch) for _ in range(nblocks)])
        self.head = nn.Sequential(nn.Conv2d(ch, ch, 3, padding=1, bias=False), nn.BatchNorm2d(ch), nn.ReLU(),
                                  nn.Conv2d(ch, 64, 1))
    def forward(self, x):
        z = self.head(self.tower(self.stem(x)))          # B x 64(to) x 8 x 8(from square = row*8+col)
        return z.flatten(2).transpose(1, 2).flatten(1)   # B x 4096, index = from*64 + to

class ChessCNNv1(nn.Module):
    """v1 / proposal architecture: plain conv blocks, 1x1 conv to 32 channels, FC layer to 4096 logits."""
    def __init__(self, cin=17 + NB, ch=64, nblocks=6, hch=32):
        super().__init__()
        def blk(i, o): return [nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU()]
        layers = blk(cin, ch)
        for _ in range(nblocks): layers += blk(ch, ch)
        self.body = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.Conv2d(ch, hch, 1), nn.ReLU(), nn.Flatten(), nn.Linear(hch * 64, 4096))
    def forward(self, x): return self.head(self.body(x))

class LinearBaseline(nn.Module):
    """score(move) = a[from-square] + b[to-square], both linear in the flattened 22x8x8 input.
    Trained with the same legal-move masking and cross-entropy as the CNNs."""
    def __init__(self, cin=17 + NB):
        super().__init__(); self.lin = nn.Linear(cin * 64, 128)
    def forward(self, x):
        z = self.lin(x.flatten(1))
        return (z[:, :64, None] + z[:, None, 64:]).flatten(1)

def n_params(m): return sum(p.numel() for p in m.parameters())

@torch.no_grad()
def predict(model, D, cond=True, band_override=None, shuffle_band=False, bs=4096):
    """Per-position arrays: hit1, hit3, nll (numpy)."""
    model.eval(); n = len(D["y"]); H1, H3, NLL = [], [], []
    g = torch.Generator(device="cpu").manual_seed(123)
    perm = torch.randperm(n, generator=g).to(device) if shuffle_band else None
    for i in range(0, n, bs):
        sl = slice(i, i + bs)
        band = D["band"][sl]
        if shuffle_band: band = D["band"][perm[sl]]
        if band_override is not None: band = torch.full_like(band, band_override)
        lg = logits_for(model, D["X"][sl], band, D["legal"][sl], cond)
        y = D["y"][sl]
        top3 = lg.topk(3, 1).indices
        H1.append((top3[:, 0] == y).cpu()); H3.append((top3 == y[:, None]).any(1).cpu())
        NLL.append(F.cross_entropy(lg, y, reduction="none").cpu())
    return dict(hit1=torch.cat(H1).numpy().astype(np.float32), hit3=torch.cat(H3).numpy().astype(np.float32),
                nll=torch.cat(NLL).numpy())

def summarize(p, band):
    band = np.asarray(band)
    return dict(loss=float(p["nll"].mean()), top1=float(p["hit1"].mean()), top3=float(p["hit3"].mean()),
                per_band_top1=[float(p["hit1"][band == b].mean()) for b in range(NB)],
                per_band_loss=[float(p["nll"][band == b].mean()) for b in range(NB)])

def train(model, cond=True, epochs=EPOCHS, lr=LR, wd=WD, name="model", seed=SEED):
    torch.manual_seed(seed)
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    n = len(Dtr["y"]); steps = epochs * ((n + BATCH - 1) // BATCH)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=steps)   # cosine decay to 0 over training
    scaler = torch.amp.GradScaler(device.type, enabled=USE_AMP)
    hist = dict(train_loss=[], val_loss=[], val_top1=[]); best = (1e9, None, 0)
    for ep in range(epochs):
        model.train(); perm = torch.randperm(n, device=device); tot = 0.0; t0 = time.time()
        for i in range(0, n, BATCH):
            idx = perm[i:i + BATCH]
            lg = logits_for(model, Dtr["X"][idx], Dtr["band"][idx], Dtr["legal"][idx], cond)
            loss = F.cross_entropy(lg, Dtr["y"][idx])
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            sched.step(); tot += loss.item() * len(idx)
        v = summarize(predict(model, Dva, cond), va["band"])
        hist["train_loss"].append(tot / n); hist["val_loss"].append(v["loss"]); hist["val_top1"].append(v["top1"])
        if v["loss"] < best[0]: best = (v["loss"], {k: t.detach().clone() for k, t in model.state_dict().items()}, ep + 1)
        print(f"[{name}] epoch {ep+1}/{epochs} train {tot/n:.3f} val {v['loss']:.3f} val top1 {v['top1']:.3f} ({time.time()-t0:.0f}s)")
    model.load_state_dict(best[1])      # early stopping: keep best validation loss
    hist["best_epoch"] = best[2]
    return hist

def fit(model, name, cond=True, seed=SEED):
    """Train, or load the finished model from the checkpoint folder if this run already completed."""
    path = os.path.join(CKPT, f"{name}.pt")
    if os.path.exists(path):
        ck = torch.load(path, map_location=device)
        model.to(device).load_state_dict(ck["state"]); model.eval()
        print(f"[{name}] loaded from checkpoint (best epoch {ck['hist']['best_epoch']})")
        return ck["hist"]
    t0 = time.time()
    hist = train(model, cond=cond, name=name, seed=seed)
    torch.save(dict(state=model.state_dict(), hist=hist), path + ".tmp")
    os.replace(path + ".tmp", path)
    print(f"[{name}] done in {(time.time()-t0)/60:.1f} min, saved | total elapsed {(time.time()-T_START)/60:.0f} min")
    return hist

## 4. Baselines: capture heuristic (no learning) and linear model

In [ ]:
# Baseline 1: hand-coded heuristic, no learning. Capture the most valuable piece (cheapest attacker on ties),
# otherwise play a uniformly random legal move. Scored on the full test set.
VAL = np.array([1, 3, 3, 5, 9, 0])
def heuristic_eval(d, seed=SEED):
    rng = np.random.RandomState(seed)
    hits = np.zeros(NB); cnt = np.zeros(NB); n_cap = 0
    for i in range(len(d["y"])):
        x = d["X"][i]; legal = d["legal"][i]; legal = legal[legal >= 0]
        opp = sum(x[6 + k].reshape(64).astype(int) * VAL[k] for k in range(6))
        own = sum(x[k].reshape(64).astype(int) * VAL[k] for k in range(6))
        frm, to = legal // 64, legal % 64
        gain = opp[to]
        if gain.max() > 0:
            n_cap += 1
            cand = np.nonzero(gain == gain.max())[0]
            cand = cand[own[frm[cand]] == own[frm[cand]].min()]
            pick = legal[rng.choice(cand)]
        else:
            pick = legal[rng.randint(len(legal))]
        b = int(d["band"][i]); cnt[b] += 1; hits[b] += (pick == d["y"][i])
    return dict(top1=float(hits.sum() / cnt.sum()), per_band_top1=(hits / np.maximum(cnt, 1)).tolist(),
                capture_available=n_cap / len(d["y"]))

def random_legal_top1(d):
    return float((1.0 / (d["legal"] >= 0).sum(1)).mean())

heur = heuristic_eval(te)
rand_ref = random_legal_top1(te)
print("heuristic top1:", round(heur["top1"], 4), "| per band:", np.round(heur["per_band_top1"], 3),
      "| capture available in", round(heur["capture_available"], 3), "| random-legal reference:", round(rand_ref, 4))

# Baseline 2: linear model on the flattened input
lin = LinearBaseline(); lin_hist = fit(lin, "linear")
lin_p = predict(lin, Dte); lin_te = summarize(lin_p, te["band"]); print("linear test:", lin_te["top1"], lin_te["top3"])

## 5. CNNs: v1 (proposal architecture) and the primary residual CNN, with and without band input, over seeds

In [ ]:
v1 = ChessCNNv1(); v1_hist = fit(v1, "v1-fc-head")
v1_p = predict(v1, Dte); v1_te = summarize(v1_p, te["band"])
print("v1 test:", v1_te["top1"], v1_te["top3"], "| best epoch", v1_hist["best_epoch"])

runs = {"band": [], "noband": []}   # each: dict(model, hist, p)
for s in range(N_SEEDS):
    for key, cond in (("band", True), ("noband", False)):
        torch.manual_seed(s); m = ChessResNet()
        h = fit(m, f"resnet-{key}-seed{s}", cond=cond, seed=s)
        p = predict(m, Dte, cond=cond)
        runs[key].append(dict(model=m, hist=h, p=p, seed=s))
        print(f"  -> {key} seed {s}: test top1 {p['hit1'].mean():.4f} top3 {p['hit3'].mean():.4f} loss {p['nll'].mean():.3f}")
cnn_params = n_params(runs["band"][0]["model"])
print("primary CNN parameters:", f"{cnn_params:,}", "| v1 parameters:", f"{n_params(v1):,}")
torch.save(runs["band"][0]["model"].state_dict(), f"{OUT}/resnet_band_seed0.pt")

def seed_stats(key, band=None, preds_key="p"):
    """Mean and std over seeds of top1/top3/loss, plus seed-averaged per-band top1."""
    S = [summarize(r[preds_key], band if band is not None else te["band"]) for r in runs[key]]
    agg = {k: float(np.mean([s[k] for s in S])) for k in ("top1", "top3", "loss")}
    agg.update({k + "_sd": float(np.std([s[k] for s in S], ddof=1)) if len(S) > 1 else 0.0 for k in ("top1", "top3", "loss")})
    agg["per_band_top1"] = np.mean([s["per_band_top1"] for s in S], 0).tolist()
    agg["per_band_top1_sd"] = (np.std([s["per_band_top1"] for s in S], 0, ddof=1) if len(S) > 1 else np.zeros(NB)).tolist()
    agg["per_seed_top1"] = [s["top1"] for s in S]
    return agg

cnn_te, nc_te = seed_stats("band"), seed_stats("noband")
print("PRIMARY band:   ", {k: round(v, 4) for k, v in cnn_te.items() if not isinstance(v, list)})
print("PRIMARY no band:", {k: round(v, 4) for k, v in nc_te.items() if not isinstance(v, list)})

## 6. Does the model use the rating band?
(a) shuffled band input, (b) paired band-vs-no-band gain with a game-level bootstrap CI,
(c) band-X-tested-on-band-Y matrix, (d) average change in predicted distribution when only the band changes.

In [ ]:
def game_bootstrap(vals, gid, n=2000, seed=0):
    """95% CI of the mean of per-position values, resampling whole games (positions in a game are correlated)."""
    _, inv = np.unique(gid, return_inverse=True)
    s = np.bincount(inv, weights=vals); c = np.bincount(inv).astype(float)
    rng = np.random.RandomState(seed); G = len(s); means = np.empty(n)
    for k in range(n):
        idx = rng.randint(G, size=G); means[k] = s[idx].sum() / c[idx].sum()
    return [float(v) for v in np.percentile(means, [2.5, 97.5])]

# (a) shuffled band input (seed-averaged)
shuf_top1 = float(np.mean([predict(r["model"], Dte, shuffle_band=True)["hit1"].mean() for r in runs["band"]]))
# (b) paired gain: average each position's hit over seeds, then band minus no-band, bootstrap over test games
h_band = np.mean([r["p"]["hit1"] for r in runs["band"]], 0); h_nc = np.mean([r["p"]["hit1"] for r in runs["noband"]], 0)
l_band = np.mean([r["p"]["nll"] for r in runs["band"]], 0); l_nc = np.mean([r["p"]["nll"] for r in runs["noband"]], 0)
gain_top1 = float((h_band - h_nc).mean()); gain_top1_ci = game_bootstrap(h_band - h_nc, te["gid"])
gain_loss = float((l_nc - l_band).mean()); gain_loss_ci = game_bootstrap(l_nc - l_band, te["gid"])
cnn_top1_ci = game_bootstrap(h_band, te["gid"])
print(f"shuffled-band top1 {shuf_top1:.4f} vs true band {cnn_te['top1']:.4f}")
print(f"band gain top1 {100*gain_top1:+.2f} pp, 95% CI {np.round(100*np.array(gain_top1_ci), 2)}")
print(f"band gain loss {gain_loss:+.4f} nats, 95% CI {np.round(gain_loss_ci, 4)}")

# (c) cross-band matrix: rows = true band of the test position, cols = band given to the model; seed-averaged
M_top1 = np.zeros((NB, NB)); M_loss = np.zeros((NB, NB))
for r in runs["band"]:
    for bi in range(NB):
        p = predict(r["model"], Dte, band_override=bi)
        for bt in range(NB):
            m = te["band"] == bt
            M_top1[bt, bi] += p["hit1"][m].mean() / len(runs["band"]); M_loss[bt, bi] += p["nll"][m].mean() / len(runs["band"])
off = ~np.eye(NB, dtype=bool)
diag_adv_top1 = float(np.mean([M_top1[b, b] - M_top1[b, off[b]].mean() for b in range(NB)]))
diag_adv_loss = float(np.mean([M_loss[b, off[b]].mean() - M_loss[b, b] for b in range(NB)]))
best_col = M_loss.argmin(1)
print("cross-band top1 (%):\n", np.round(100 * M_top1, 1))
print("cross-band loss:\n", np.round(M_loss, 3))
print(f"matched minus mismatched band: top1 {100*diag_adv_top1:+.2f} pp, loss {diag_adv_loss:+.4f} nats;"
      f" lowest-loss input band per true band: {best_col.tolist()}")

# (d) total-variation distance between predictions with band 1100-1300 vs 1900-2100, same position
@torch.no_grad()
def probs_batch(model, D, idx, band_value, cond=True):
    model.eval(); idx = torch.as_tensor(idx, device=device)
    b = torch.full((len(idx),), band_value, device=device, dtype=torch.long)
    return F.softmax(logits_for(model, D["X"][idx], b, D["legal"][idx], cond), 1)

rng = np.random.RandomState(1); tv_idx = rng.permutation(len(te["y"]))[:min(4000, len(te["y"]))]
m0 = runs["band"][0]["model"]
tv = (0.5 * (probs_batch(m0, Dte, tv_idx, 0) - probs_batch(m0, Dte, tv_idx, NB - 1)).abs().sum(1)).cpu().numpy()
mean_tv, max_tv = float(tv.mean()), float(tv.max())
print(f"TV(band 0 vs band 4) over {len(tv_idx)} test positions: mean {mean_tv:.3f}, median {np.median(tv):.3f}, max {max_tv:.3f}")

# (e) player leakage: accuracy on test positions whose players never appear in training
unseen_top1 = float(h_band[~te_seen].mean()) if (~te_seen).any() else float("nan")
seen_top1 = float(h_band[te_seen].mean()) if te_seen.any() else float("nan")
print(f"primary top1 on test games with a player seen in training: {seen_top1:.4f} ({te_seen.mean():.1%} of test) | "
      f"unseen players only: {unseen_top1:.4f}")

## 7. Figures: learning curves, per-band accuracy + cross-band matrix, qualitative examples

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(FIG_W, 2.25))
ep = np.arange(1, EPOCHS + 1)
hb, hv = runs["band"][0]["hist"], v1_hist
ax[0].plot(ep, hb["train_loss"], color=C_CNN, ls="--", label="ResNet train")
ax[0].plot(ep, hb["val_loss"], color=C_CNN, label="ResNet val")
ax[0].plot(ep, hv["train_loss"], color=C_V1, ls="--", label="v1 FC-head train")
ax[0].plot(ep, hv["val_loss"], color=C_V1, label="v1 FC-head val")
ax[0].plot(ep, lin_hist["val_loss"], color=C_LIN, ls=":", label="Linear val")
for h, c in ((hb, C_CNN), (hv, C_V1)):
    ax[0].plot(h["best_epoch"], h["val_loss"][h["best_epoch"] - 1], "o", color=c, ms=4)
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("cross-entropy (nats)"); ax[0].set_title("(a) Loss (dot = kept checkpoint)")
ax[0].set_xticks(ep)
ax[1].plot(ep, 100 * np.array(hb["val_top1"]), color=C_CNN, label="ResNet, band input")
ax[1].plot(ep, 100 * np.array(runs["noband"][0]["hist"]["val_top1"]), color=C_NC, ls="-.", label="ResNet, no band")
ax[1].plot(ep, 100 * np.array(hv["val_top1"]), color=C_V1, label="v1 FC-head")
ax[1].plot(ep, 100 * np.array(lin_hist["val_top1"]), color=C_LIN, ls=":", label="Linear")
ax[1].axhline(100 * heur["top1"], color=C_HEUR, ls="--", lw=1, label="Capture heuristic")
ax[1].set_xlabel("epoch"); ax[1].set_ylabel("val top-1 (%)"); ax[1].set_title("(b) Validation top-1 accuracy")
ax[1].set_xticks(ep)
h0, l0 = ax[0].get_legend_handles_labels(); h1, l1 = ax[1].get_legend_handles_labels()
ax[0].legend(h0, l0, loc="upper center", bbox_to_anchor=(0.5, -0.3), ncol=2, frameon=False, fontsize=6.5)
ax[1].legend(h1, l1, loc="upper center", bbox_to_anchor=(0.5, -0.3), ncol=2, frameon=False, fontsize=6.5)
plt.tight_layout(pad=0.3); plt.savefig(f"{OUT}/fig_curves.png", bbox_inches="tight"); plt.savefig(f"{OUT}/fig_curves.pdf", bbox_inches="tight"); plt.close()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(FIG_W, 2.3), gridspec_kw=dict(width_ratios=[1.45, 1]))
w = .16; xs = np.arange(NB)
bars = [("Heuristic", heur["per_band_top1"], None, C_HEUR), ("Linear", lin_te["per_band_top1"], None, C_LIN),
        ("v1 FC-head", v1_te["per_band_top1"], None, C_V1),
        ("ResNet, no band", nc_te["per_band_top1"], nc_te["per_band_top1_sd"], C_NC),
        ("ResNet, band", cnn_te["per_band_top1"], cnn_te["per_band_top1_sd"], C_CNN)]
for j, (lab, v, e, c) in enumerate(bars):
    ax[0].bar(xs + (j - 2) * w, 100 * np.array(v), w, yerr=None if e is None else 100 * np.array(e),
              color=c, label=lab, error_kw=dict(lw=.6, capsize=1))
band_ticks(ax[0])
ax[0].set_ylabel("test top-1 (%)"); ax[0].set_title("(a) Test top-1 by band\n(ResNet: mean $\\pm$ sd over seeds)")
fig.legend(*ax[0].get_legend_handles_labels(), loc="lower center", bbox_to_anchor=(0.5, 0.0), ncol=5,
           frameon=False, fontsize=6.5, handlelength=1.2, columnspacing=1.0)
dev = 100 * (M_top1 - M_top1.mean(1, keepdims=True))
lim = max(abs(dev).max(), 0.1)
im = ax[1].imshow(dev, cmap="RdBu", vmin=-lim, vmax=lim)
for i in range(NB):
    for j in range(NB):
        ax[1].text(j, i, f"{100*M_top1[i, j]:.1f}", ha="center", va="center", fontsize=5.5,
                   fontweight="bold" if i == j else "normal")
band_ticks(ax[1], "x", 5.5); band_ticks(ax[1], "y", 5.5)
ax[1].set_xlabel("band given to the model"); ax[1].set_ylabel("true band of position")
ax[1].set_title("(b) Top-1 (%) with the band\ninput swapped")
cb = plt.colorbar(im, ax=ax[1], fraction=.046, pad=.04); cb.set_label("pp vs. row mean", fontsize=6); cb.ax.tick_params(labelsize=6)
plt.tight_layout(pad=0.3, rect=(0, 0.12, 1, 1)); plt.savefig(f"{OUT}/fig_bands.png", bbox_inches="tight"); plt.savefig(f"{OUT}/fig_bands.pdf", bbox_inches="tight"); plt.close()

In [ ]:
@torch.no_grad()
def probs_for(model, D, i, band):
    return probs_batch(model, D, [i], band)[0].cpu().numpy()

best_i = int(tv_idx[tv.argmax()])   # largest band effect among the sampled test positions (cherry-picked on purpose)
with torch.no_grad():
    k = min(3000, len(Dte["y"])); m0.eval()
    pr = F.softmax(logits_for(m0, Dte["X"][:k], Dte["band"][:k], Dte["legal"][:k]), 1)
    conf, pred = pr.max(1); correct = pred == Dte["y"][:k]
ok_i = int(torch.nonzero(correct & (conf > .6))[0]) if (correct & (conf > .6)).any() else 0
bad_i = int(torch.nonzero(~correct & (conf > .5))[0]) if (~correct & (conf > .5)).any() else 1

fig = plt.figure(figsize=(FIG_W, 3.7))
gs = fig.add_gridspec(2, 3, height_ratios=[1, 1], width_ratios=[1, 1, 1.05])
for kk, (i, title) in enumerate([(ok_i, "(a) Model right"), (bad_i, "(b) Model wrong")]):
    ax = fig.add_subplot(gs[0, kk]); x = te["X"][i]; y = int(te["y"][i])
    p = probs_for(m0, Dte, i, int(te["band"][i])); top = int(p.argmax())
    draw_board(ax, x, [(y, C_CNN, "-"), (top, "#d62728", "--")] if top != y else [(y, C_CNN, "-")], fs=10)
    ax.set_title(f"{title}, band {BAND_NAMES[int(te['band'][i])]}\nplayed {mv_name(y)}; model {mv_name(top)} ({p[top]:.0%})", fontsize=7)
ax = fig.add_subplot(gs[0, 2]); ax.axis("off")
ax.text(0, .5, "Green arrow: move the human played.\nRed dashed: model's top choice.\n"
        "Mover is always at the bottom\n(boards are flipped for Black).\n\n(c)-(d): one position, same model,\n"
        f"only the band input changes.\nLargest effect among {len(tv_idx):,} test\npositions; the average effect is\n"
        f"much smaller (mean TV {mean_tv:.2f} vs {max_tv:.2f}).", fontsize=6.5, va="center")
x = te["X"][best_i]; y = int(te["y"][best_i])
ax = fig.add_subplot(gs[1, 0]); draw_board(ax, x, [(y, C_CNN, "-")], fs=10)
ax.set_title(f"(c) Largest band effect\nband {BAND_NAMES[int(te['band'][best_i])]} played {mv_name(y)}", fontsize=7)
P = np.stack([probs_for(m0, Dte, best_i, b) for b in range(NB)])
topmoves = list(np.argsort(-P.max(0))[:4])
ax = fig.add_subplot(gs[1, 1:])
cols = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]
for j, mvi in enumerate(topmoves):
    ax.plot(range(NB), P[:, mvi], "o-", color=cols[j], ms=3, lw=1.2, label=mv_name(int(mvi)))
ax.set_xticks(range(NB)); ax.set_xticklabels(BAND_NAMES, fontsize=6.5); ax.set_ylabel("predicted probability")
ax.set_xlabel("rating band given to the model"); ax.set_ylim(0, None)
ax.legend(fontsize=6.5, title="move", title_fontsize=6.5, frameon=False, loc="center left", bbox_to_anchor=(1, .5))
ax.set_title("(d) Predicted probability vs. band input", fontsize=7)
plt.tight_layout(pad=0.3); plt.savefig(f"{OUT}/fig_qualitative.png", bbox_inches="tight"); plt.savefig(f"{OUT}/fig_qualitative.pdf", bbox_inches="tight"); plt.close()

## 8. Final-test data: a completely different month (never used for training/validation/tuning)

In [ ]:
new_raw, new_stats = cached_dataset("dataset_newmonth", TEST_MONTH, TEST_GAMES_PER_BAND, SEED + 1)
Dnew = to_dev(new_raw)
for key in runs:
    for r in runs[key]: r["p_new"] = predict(r["model"], Dnew, cond=(key == "band"))
cnn_new, nc_new = seed_stats("band", band=new_raw["band"], preds_key="p_new"), seed_stats("noband", band=new_raw["band"], preds_key="p_new")
lin_new = summarize(predict(lin, Dnew), new_raw["band"]); v1_new = summarize(predict(v1, Dnew), new_raw["band"])
heur_new = heuristic_eval(new_raw)
new_seen = seen_mask(new_raw)
print(f"NEW MONTH | ResNet {cnn_new['top1']:.4f}+-{cnn_new['top1_sd']:.4f} | no band {nc_new['top1']:.4f} | "
      f"v1 {v1_new['top1']:.4f} | Linear {lin_new['top1']:.4f} | Heuristic {heur_new['top1']:.4f} | "
      f"new-month positions with a player seen in training: {new_seen.mean():.1%}")

## 9. Save everything

In [ ]:
res = dict(
    config=dict(train_month=TRAIN_MONTH, test_month=TEST_MONTH, games_per_band=GAMES_PER_BAND, keep_prob=KEEP_PROB,
                pos_per_game=POS_PER_GAME, epochs=EPOCHS, batch=BATCH, lr=LR, weight_decay=WD, n_seeds=N_SEEDS,
                ch=CH, nblocks=NBLOCKS, device=str(device), amp=USE_AMP, synthetic=SYNTHETIC),
    data=dict(scanned=train_stats["scanned"], games_per_band=games_per_band, pos_per_band=pos_per_band,
              first_game_utc=train_stats["first_game_utc"], last_game_utc=train_stats["last_game_utc"],
              total_positions=int(len(train_raw["y"])), n_train=len(tr["y"]), n_val=len(va["y"]), n_test=len(te["y"]),
              avg_legal=avg_legal, test_frac_player_seen=float(te_seen.mean()),
              new_month_positions=int(len(new_raw["y"])), new_month_games=new_stats["games_per_band"],
              new_month_scanned=new_stats["scanned"], new_first_game_utc=new_stats["first_game_utc"],
              new_last_game_utc=new_stats["last_game_utc"], new_frac_player_seen=float(new_seen.mean())),
    heuristic=heur, random_legal_top1=rand_ref, linear=lin_te, v1=v1_te, cnn=cnn_te, cnn_no_band=nc_te,
    cnn_top1_ci95=cnn_top1_ci, cnn_shuffled_band_top1=shuf_top1,
    band_gain=dict(top1=gain_top1, top1_ci95=gain_top1_ci, loss=gain_loss, loss_ci95=gain_loss_ci),
    cross_band=dict(top1=M_top1.tolist(), loss=M_loss.tolist(), matched_adv_top1=diag_adv_top1,
                    matched_adv_loss=diag_adv_loss, lowest_loss_input_band=best_col.tolist()),
    band_effect_tv=dict(mean=mean_tv, median=float(np.median(tv)), max=max_tv, n=int(len(tv_idx))),
    player_leakage=dict(top1_seen=seen_top1, top1_unseen=unseen_top1),
    cnn_params=cnn_params, v1_params=n_params(v1), linear_params=n_params(lin),
    new_month=dict(cnn=cnn_new, cnn_no_band=nc_new, v1=v1_new, linear=lin_new, heuristic=heur_new),
    history=dict(linear=lin_hist, v1=v1_hist, cnn=[r["hist"] for r in runs["band"]],
                 cnn_no_band=[r["hist"] for r in runs["noband"]]))
json.dump(res, open(f"{OUT}/results.json", "w"), indent=1, default=float)

def pct(v): return f"{100*v:.1f}"
def pp(v): return f"{100*v:+.1f}"
def big(v): return f"{v:,}".replace(",", "{,}")
with open(f"{OUT}/numbers.tex", "w") as f:
    def mac(name, val): f.write(f"\\newcommand{{\\{name}}}{{{val}}}\n")
    # data
    mac("nTotal", big(len(train_raw["y"]))); mac("nTrain", big(len(tr["y"]))); mac("nVal", big(len(va["y"])))
    mac("nTest", big(len(te["y"]))); mac("nNew", big(len(new_raw["y"]))); mac("nGamesBand", big(min(games_per_band)))
    mac("nGamesNewBand", big(min(new_stats["games_per_band"])))
    mac("nScanned", big(train_stats["scanned"])); mac("nScannedNew", big(new_stats["scanned"]))
    mac("avgLegal", f"{avg_legal:.0f}"); mac("keepProb", f"{KEEP_PROB:g}")
    mac("spanStart", train_stats["first_game_utc"]); mac("spanEnd", train_stats["last_game_utc"])
    mac("spanNewStart", new_stats["first_game_utc"]); mac("spanNewEnd", new_stats["last_game_utc"])
    mac("seenFrac", pct(te_seen.mean())); mac("seenFracNew", pct(new_seen.mean()))
    # models
    mac("cnnParams", f"{cnn_params/1e6:.1f}M"); mac("vOneParams", f"{n_params(v1)/1e6:.1f}M"); mac("linParams", big(n_params(lin)))
    mac("nSeeds", N_SEEDS); mac("nEpochs", EPOCHS); mac("chW", CH); mac("nBlocks", NBLOCKS)
    mac("bestEpCnn", runs["band"][0]["hist"]["best_epoch"]); mac("bestEpVOne", v1_hist["best_epoch"])
    # test results
    mac("topOneCnn", pct(cnn_te["top1"])); mac("topOneCnnSd", pct(cnn_te["top1_sd"]))
    mac("topThreeCnn", pct(cnn_te["top3"])); mac("lossCnn", f"{cnn_te['loss']:.2f}")
    mac("topOneCnnLo", pct(cnn_top1_ci[0])); mac("topOneCnnHi", pct(cnn_top1_ci[1]))
    mac("topOneNc", pct(nc_te["top1"])); mac("topOneNcSd", pct(nc_te["top1_sd"]))
    mac("topThreeNc", pct(nc_te["top3"])); mac("lossNc", f"{nc_te['loss']:.2f}")
    mac("topOneVOne", pct(v1_te["top1"])); mac("topThreeVOne", pct(v1_te["top3"])); mac("lossVOne", f"{v1_te['loss']:.2f}")
    mac("topOneLin", pct(lin_te["top1"])); mac("topThreeLin", pct(lin_te["top3"])); mac("lossLin", f"{lin_te['loss']:.2f}")
    mac("topOneHeur", pct(heur["top1"])); mac("topOneRand", pct(rand_ref)); mac("capAvail", pct(heur["capture_available"]))
    mac("heurLow", pct(heur["per_band_top1"][0])); mac("heurHigh", pct(heur["per_band_top1"][-1]))
    # band analysis
    mac("topOneCnnShuf", pct(shuf_top1))
    mac("bandGain", pp(gain_top1)); mac("bandGainLo", pp(gain_top1_ci[0])); mac("bandGainHi", pp(gain_top1_ci[1]))
    mac("bandGainLoss", f"{gain_loss:+.3f}"); mac("bandGainLossLo", f"{gain_loss_ci[0]:+.3f}"); mac("bandGainLossHi", f"{gain_loss_ci[1]:+.3f}")
    mac("diagAdvTop", pp(diag_adv_top1)); mac("diagAdvLoss", f"{diag_adv_loss:+.3f}")
    mac("meanTV", f"{mean_tv:.2f}"); mac("bandTV", f"{max_tv:.2f}"); mac("nTV", big(len(tv_idx)))
    mac("topOneSeen", pct(seen_top1)); mac("topOneUnseen", pct(unseen_top1))
    # new month
    mac("newCnn", pct(cnn_new["top1"])); mac("newCnnSd", pct(cnn_new["top1_sd"])); mac("newCnnThree", pct(cnn_new["top3"]))
    mac("newNc", pct(nc_new["top1"])); mac("newVOne", pct(v1_new["top1"])); mac("newLin", pct(lin_new["top1"]))
    mac("newHeur", pct(heur_new["top1"]))
print(open(f"{OUT}/numbers.tex").read())

shutil.make_archive("progress_outputs", "zip", OUT)
print("\n==== SEND progress_outputs.zip BACK (or paste results.json) ====")
try:
    from google.colab import files; files.download("progress_outputs.zip")
except Exception:
    pass